# Chapter 14 — Decision Trees

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/winnertechserv/aimlgybs/blob/main/part2_classic_ml/ch14_decision_trees/ch14_notebook.ipynb)

Companion notebook for **Chapter 14** of *AI-Machine Learning: Get Your Base Strong*. Read the chapter first, then run each cell and compare your output with the output printed in the book (shown below each cell).


In [ ]:
# Optional: install the exact library versions used for the outputs in the book.
# Google Colab already has these libraries; newer versions may change a few digits.
# !pip install -q numpy==2.5.3 pandas==3.0.5 scipy==1.18.1 scikit-learn==1.9.1 matplotlib==3.11.2 pillow==12.3.0

## Key takeaways

- A decision tree is a sequence of questions; each split should leave the child nodes purer than the parent.
- Entropy and Gini both measure impurity, and information gain picks the split that reduces it most.
- A tree grown without limits will memorize the training set: stop early using depth or minimum-samples rules.
- Occam's Razor applies — prefer the simpler tree when two trees explain the data equally well.
- Trees are highly interpretable: you can read the decision rules straight off the diagram.

## Example 1: Entropy Impurity & Information Gain

This is the most basic calculation in decision tree construction — manual entropy and information gain computation. We use a small worked example: a node with 10 examples (4 from C1, 4 from C2, 2 from C3) and two competing queries. For each query, we compute entropy of the parent node and both child nodes, then calculate information gain. Query 1 wins (ΔI = 0.3219 > 0.1568) so it gets selected. The bottom section shows entropy for special cases — a pure node has entropy 0 (no uncertainty), while equal class distribution gives maximum entropy.

In [1]:
import numpy as np

def entropy(counts):
    total = sum(counts)
    if total == 0: return 0
    probs = [c / total for c in counts if c > 0]
    return -sum(p * np.log2(p) for p in probs) + 0.0   # + 0.0 turns -0.0 into 0.0

def info_gain(parent, left, right):
    n = sum(parent); nl = sum(left); nr = sum(right)
    return entropy(parent) - (nl/n)*entropy(left) - (nr/n)*entropy(right)

# Node Nd: 4 from C1, 4 from C2, 2 from C3
parent = [4, 4, 2]
print(f"Parent Node: {parent[0]} C1, {parent[1]} C2, {parent[2]} C3")
print(f"Entropy i(Nd) = {entropy(parent):.4f}\n")

# Query 1
left1, right1 = [3, 1, 0], [1, 3, 2]
print(f"Query 1: Left={left1}  Right={right1}")
print(f"  i(NdL)={entropy(left1):.4f}  i(NdR)={entropy(right1):.4f}")
print(f"  Information Gain = {info_gain(parent, left1, right1):.4f}")

# Query 2
left2, right2 = [2, 1, 0], [2, 3, 2]
print(f"\nQuery 2: Left={left2}  Right={right2}")
print(f"  i(NdL)={entropy(left2):.4f}  i(NdR)={entropy(right2):.4f}")
print(f"  Information Gain = {info_gain(parent, left2, right2):.4f}")

ig1 = info_gain(parent, left1, right1)
ig2 = info_gain(parent, left2, right2)
print(f"\nDecision: Query 1 wins ({ig1:.4f} > {ig2:.4f}) -> SELECT Query 1")

print("\n--- Entropy for special cases ---")
print(f"  Pure node [10, 0]  : entropy = {entropy([10,0]):.4f}  (pure, no uncertainty)")
print(f"  50/50 split [5,5]  : entropy = {entropy([5,5]):.4f}  (maximum, 2 classes)")
print(f"  Equal [1,1,1]      : entropy = {entropy([1,1,1]):.4f}  (maximum, 3 classes)")

Parent Node: 4 C1, 4 C2, 2 C3
Entropy i(Nd) = 1.5219

Query 1: Left=[3, 1, 0]  Right=[1, 3, 2]
  i(NdL)=0.8113  i(NdR)=1.4591
  Information Gain = 0.3219

Query 2: Left=[2, 1, 0]  Right=[2, 3, 2]
  i(NdL)=0.9183  i(NdR)=1.5567
  Information Gain = 0.1568

Decision: Query 1 wins (0.3219 > 0.1568) -> SELECT Query 1

--- Entropy for special cases ---
  Pure node [10, 0]  : entropy = 0.0000  (pure, no uncertainty)
  50/50 split [5,5]  : entropy = 1.0000  (maximum, 2 classes)
  Equal [1,1,1]      : entropy = 1.5850  (maximum, 3 classes)


## Example 2: CART Decision Tree with scikit-learn (Fruit Classifier)

We train a real CART decision tree on a small fruit classification example. Each fruit is described by 4 categorical features encoded as integers: Color (0=Green, 1=Yellow, 2=Red), Size (0=Small, 1=Medium, 2=Big), Shape (0=Round, 1=NonRound), Taste (0=NonSweet, 1=Sweet). The export_text output shows the actual tree that CART built — it's a human-readable flowchart you can follow manually. We then test it on three new fruit descriptions to check predictions. Scikit-learn trees need numbers, which is why the categories are encoded as integers first. Figure 14.1 draws this same tree. Notice that it never asks about Shape: color, size and taste are enough to tell these twelve fruits apart.

In [2]:
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.metrics import accuracy_score
import numpy as np

# Encoding: Color:0=Green,1=Yellow,2=Red | Size:0=Small,1=Med,2=Big
# Shape:0=Round,1=NonRound | Taste:0=NonSweet,1=Sweet
X = np.array([
    [0,2,0,0],[0,2,0,0],             # Watermelon
    [2,1,0,1],[0,1,0,0],[2,1,0,1],  # Apple
    [1,1,0,0],[1,2,0,0],             # Lemon, Grapefruit
    [1,0,1,0],[1,0,1,0],             # Banana
    [2,0,0,1],                        # Cherry
    [2,0,0,0],[2,0,1,0],             # Grape
])
y = np.array([6,6, 0,0,0, 5,4, 1,1, 2, 3,3])
class_names   = ['Apple','Banana','Cherry','Grape','Grapefruit','Lemon','Watermelon']
feature_names = ['Color','Size','Shape','Taste']

clf = DecisionTreeClassifier(criterion='entropy', max_depth=4, random_state=42)
clf.fit(X, y)

print("=== Decision Tree (CART, criterion=entropy) ===\n")
print(export_text(clf, feature_names=feature_names, class_names=class_names))

print(f"Training Accuracy: {accuracy_score(y, clf.predict(X))*100:.1f}%\n")

print("=== Predictions on New Fruit Descriptions ===")
tests = {
    'Red, Small, Round, Sweet'    : [2, 0, 0, 1],
    'Yellow, Big, Round, NonSweet': [1, 2, 0, 0],
    'Green, Big, Round, NonSweet' : [0, 2, 0, 0],
}
for desc, feat in tests.items():
    pred = clf.predict([feat])[0]
    print(f"  {desc:<45} -> {class_names[pred]}")

=== Decision Tree (CART, criterion=entropy) ===

|--- Size <= 0.50
|   |--- Color <= 1.50
|   |   |--- class: Banana
|   |--- Color >  1.50
|   |   |--- Taste <= 0.50
|   |   |   |--- class: Grape
|   |   |--- Taste >  0.50
|   |   |   |--- class: Cherry
|--- Size >  0.50
|   |--- Size <= 1.50
|   |   |--- Taste <= 0.50
|   |   |   |--- Color <= 0.50
|   |   |   |   |--- class: Apple
|   |   |   |--- Color >  0.50
|   |   |   |   |--- class: Lemon
|   |   |--- Taste >  0.50
|   |   |   |--- class: Apple
|   |--- Size >  1.50
|   |   |--- Color <= 0.50
|   |   |   |--- class: Watermelon
|   |   |--- Color >  0.50
|   |   |   |--- class: Grapefruit

Training Accuracy: 100.0%

=== Predictions on New Fruit Descriptions ===
  Red, Small, Round, Sweet                      -> Cherry
  Yellow, Big, Round, NonSweet                  -> Grapefruit
  Green, Big, Round, NonSweet                   -> Watermelon


## Example 3: Tree Depth vs Overfitting (Occam's Razor in Action)

This example shows Occam's Razor in decision trees — the principle that simpler models are preferred when they explain the data equally well. We train the same tree on the Iris dataset with different maximum depths and compare training vs test accuracy. A very deep tree achieves 100% training accuracy (it perfectly memorizes the data) but test accuracy may not improve and can drop. The sweet spot is around depth 2-3 for Iris. We also compare Gini vs Entropy criteria, confirming they produce nearly identical results in practice — the choice rarely matters much.

In [3]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

iris = load_iris()
X, y = iris.data, iris.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

print("=== Tree Depth vs Accuracy (Iris Dataset) ===\n")
print(f"{'Max Depth':<12} {'Train Acc':>10} {'Test Acc':>10} {'Leaf Nodes':>12}")
print("-" * 48)
for depth in [1, 2, 3, 4, 5, None]:
    clf = DecisionTreeClassifier(max_depth=depth, criterion='gini', random_state=42)
    clf.fit(X_train, y_train)
    tr = accuracy_score(y_train, clf.predict(X_train))
    te = accuracy_score(y_test,  clf.predict(X_test))
    d  = str(depth) if depth else "No limit"
    print(f"  {d:<10} {tr*100:>9.1f}% {te*100:>9.1f}% {clf.get_n_leaves():>12}")

print()
print("Insight: Deeper trees memorize training data (Occam's Razor says stop early!)")
print()
print("=== Gini vs Entropy Impurity Criterion ===\n")
for crit in ['gini', 'entropy']:
    clf = DecisionTreeClassifier(max_depth=3, criterion=crit, random_state=42)
    clf.fit(X_train, y_train)
    te = accuracy_score(y_test, clf.predict(X_test))
    print(f"  criterion='{crit}': Test={te*100:.1f}%  Leaves={clf.get_n_leaves()}")
print("\nConclusion: Gini and Entropy usually produce very similar trees in practice.")

=== Tree Depth vs Accuracy (Iris Dataset) ===

Max Depth     Train Acc   Test Acc   Leaf Nodes
------------------------------------------------
  1               64.8%      71.1%            2
  2               94.3%      97.8%            3
  3               95.2%     100.0%            5
  4               97.1%     100.0%            7
  5               99.0%     100.0%            9
  No limit       100.0%     100.0%           10

Insight: Deeper trees memorize training data (Occam's Razor says stop early!)

=== Gini vs Entropy Impurity Criterion ===

  criterion='gini': Test=100.0%  Leaves=5
  criterion='entropy': Test=97.8%  Leaves=5

Conclusion: Gini and Entropy usually produce very similar trees in practice.
